# Electricity Consumption Forecasting: LSTM vs. Traditional ML Models
**University Case Study Project (10 Marks Deliverable)**

This notebook demonstrates an end-to-end Machine Learning and Deep Learning pipeline to forecast electricity consumption using:
1. **Persistence (Naive Lag-1) Baseline**
2. **Linear Regression (Ridge)**
3. **Random Forest Regressor**
4. **Stacked Long Short-Term Memory (LSTM) Neural Network**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Add src directory to path
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_electricity_data
from src.feature_engineering import build_feature_pipeline
from src.preprocessing import prepare_lstm_data, prepare_tabular_data, inverse_transform_predictions
from src.baseline_models import evaluate_baselines
from src.lstm_model import build_lstm_model, train_lstm
from src.evaluation import calculate_metrics, compare_models
from src.predict import predict_next_step, predict_multi_step_autoregressive

sns.set_theme(style='whitegrid')
print('All libraries successfully imported!')

## 1. Dataset Loading and Initial Inspection
We use the **Open Power System Data (OPSD)** platform daily electricity consumption dataset for Germany (2006-2017).

In [ ]:
df = load_electricity_data()
print(f'Total observations: {len(df)}')
df.head()

## 2. Exploratory Data Analysis & Seasonality Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['Consumption'], label='Daily Consumption', alpha=0.5)
ax.plot(df.index, df['Consumption'].rolling(30).mean(), label='30-Day Moving Avg', color='red', linewidth=2)
ax.set_title('Historical Electricity Consumption in Germany (2006-2017)', fontweight='bold')
ax.set_ylabel('Consumption (GWh)')
ax.set_xlabel('Date')
ax.legend()
plt.show()

## 3. Weekly and Monthly Boxplots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df_copy = df.copy()
df_copy['Day'] = [day_names[d] for d in df_copy.index.dayofweek]
df_copy['Month'] = df_copy.index.month
sns.boxplot(data=df_copy, x='Day', y='Consumption', ax=ax1, palette='Blues')
ax1.set_title('Weekly Profile (Demand drops on Sat/Sun)')
sns.boxplot(data=df_copy, x='Month', y='Consumption', ax=ax2, palette='Oranges')
ax2.set_title('Monthly Profile (Winter peaks)')
plt.show()

## 4. Model Training & Pipeline Orchestration

In [ ]:
from main import run_pipeline
df_summary = run_pipeline(epochs=25)
df_summary

## 5. Inference & Future Forecasting

In [ ]:
sample_input = df['Consumption'].values[-14:]
forecast_7days = predict_multi_step_autoregressive(sample_input, horizon=7)
for i, val in enumerate(forecast_7days, 1):
    print(f'Day +{i}: {val:.2f} GWh')